In [ ]:
%config HistoryManager.enabled = False

In [ ]:
spark.sparkContext.setLogLevel("ERROR")

# Create table and insert 2 records

In [ ]:
spark.sql("CREATE TABLE IF NOT EXISTS demo.test (id int, data string) USING iceberg")

In [ ]:
spark.sql("INSERT INTO demo.test VALUES (1, 'AAA')")

In [ ]:
spark.sql("INSERT INTO demo.test VALUES (2, 'BBB')")

In [ ]:
spark.sql("SELECT * FROM demo.test").show()

# Check snapshots

In [ ]:
spark.sql("SELECT committed_at, snapshot_id, operation FROM demo.test.snapshots").show(truncate=False)

In [ ]:
target_snapshot_id = spark.sql("SELECT snapshot_id FROM demo.test.snapshots ORDER BY committed_at LIMIT 1").collect()[0][0]

df = spark.read \
    .option("snapshot-id", target_snapshot_id) \
    .table("demo.test")

df.show()

# Rollback

In [ ]:
first_snapshot_id = spark.sql("SELECT snapshot_id FROM demo.test.snapshots ORDER BY committed_at LIMIT 1").collect()[0][0]

spark.sql(f"CALL demo.system.rollback_to_snapshot('test', {first_snapshot_id})")

spark.sql("SELECT * FROM demo.test").show()